In [ ]:
!pip install --upgrade google-play-scraper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.1 MB/s eta 0:00:00


In [ ]:

!pip install --upgrade beautifulsoup4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 6.2 MB/s eta 0:00:00
  Attempting uninstall: beautifulsoup4
    Found existing installation: beautifulsoup4 4.13.5
    Uninstalling beautifulsoup4-4.13.5:
      Successfully uninstalled beautifulsoup4-4.13.5


In [ ]:
#from google_play_scraper import collection
#from google_play_scraper.constants import Collection, Category

import time
from bs4 import BeautifulSoup
import json
import pandas as pd
import requests
from bs4 import BeautifulSoup
from google_play_scraper import app, permissions


#Identified risk categories in each app's permissions
risk_categories = {
    "overlay": ["draw over other apps"],
    "background": ["run at startup", "prevent device from sleeping"],
    "bluetooth": ["pair with Bluetooth devices", "access Bluetooth settings"],
    "network": ["full network access", "view network connections"]
}


In [ ]:


def collect_app_data():
    error_apps = {}

    with open("family_apps_strict_enriched.json", "r") as f:
        data = json.load(f)

    df = pd.json_normalize(data)

    flag_dict = {}
    seen_apps = set()

    for _, row in df.iterrows():

        app_id = row.get("appId")

        if not app_id:
            continue

        if app_id in seen_apps:
            continue
        seen_apps.add(app_id)

        try:
            # Get permissions + details
            perm = permissions(app_id)
            url = row.get("privacyPolicy")
            title = row.get("title")
            genre = row.get("genre")

            #url = details.get("privacyPolicy")
            if not url:
                continue

            response = requests.get(url, timeout=3)
            soup = BeautifulSoup(response.text, "lxml")

            for tag in soup(["script", "style", "noscript"]):
                tag.decompose()

            policy_text = soup.get_text("\n").lower()
            other_permissions = perm.get("Other", [])


            flag_dict[app_id] = {
                "title": title,
                "genre": genre,
                "permissions": other_permissions,
                "policy_text": policy_text
            }

        except Exception as e:
            print(f"Error fetching {app_id}: {e}")
            error_apps[app_id] = e


    return flag_dict


In [ ]:

#Track apps that request at least one Bluetooth related permission
bluetooth_apps = []

#Track apps that request network access related permission
network_apps = []

#call data collection function
flag_dict = collect_app_data()


#for app_id, and given data from the flag_dict, set perms to the permissions of each app
for app_id, data in flag_dict.items():
    perms = data["permissions"]

    #Check for bluetooth risk category in the permissions
    if any(p in risk_categories["bluetooth"] for p in perms):
        bluetooth_apps.append(data["title"])

    if any(p in risk_categories["network"] for p in perms):
      network_apps.append(data["title"])


bluetooth_rate = len(bluetooth_apps) / len(flag_dict)



print(f"Apps that ask for bluetooth permission: {len(bluetooth_apps)} / {len(flag_dict)} = {bluetooth_rate}")
count = 0

#for title in bluetooth_apps:
    #count += 1
    #print(f"{count}. {title}")

print("\nBluetooth Permission Rate:", round(bluetooth_rate * 100, 2), "%")

Error fetching se.filimundus.pi2: HTTPSConnectionPool(host='filimundus.se', port=443): Read timed out. (read timeout=3)
Error fetching com.playhome.plus: HTTPConnectionPool(host='myplayhomeapp.com', port=80): Read timed out. (read timeout=3)
Error fetching com.ninjakiwi.btdadventuretime: HTTPConnectionPool(host='ninjakiwi.com', port=80): Read timed out. (read timeout=3)
Error fetching com.magisterapp.iceage: HTTPSConnectionPool(host='www.magisterapp.com', port=443): Read timed out. (read timeout=3)
Error fetching kila.thesmartfox: HTTPSConnectionPool(host='kila.app', port=443): Read timed out. (read timeout=3)
Error fetching com.bimiboo.playandlearn: HTTPSConnectionPool(host='bimiboo.net', port=443): Read timed out. (read timeout=3)
Error fetching com.bimiboo.coloring: HTTPSConnectionPool(host='bimiboo.net', port=443): Read timed out. (read timeout=3)
Error fetching com.forkids.family.smartkids: HTTPSConnectionPool(host='4kids.live', port=443): Max retries exceeded with url: /privacy-p

## Cell Below may be adapted or deleted in the future. I'll keep it here for now. Right now it just checks for the keyword "bluetooth", but the cell below this one identifies key bluetooth phrases not just the word bluetooth.

In [ ]:

#list of undisclosed apps (Bluetooth)
#bluetooth_undisclosed = []

network_undisclosed = []


network_rate = len(network_apps) / len(flag_dict)

print(f"Apps that ask for Network permission(s): {len(network_apps)} / {len(flag_dict)} = {network_rate}")

print(f"\nNetwork Permission Rate: {(network_rate * 100):.2f}%")


#create perms and policy variables
for app_id, data in flag_dict.items():
    perms = data["permissions"]
    policy = data["policy_text"]


    #Boolean check for bluetooth risk and if it mentions Bluetooth in the policy.
    #If not, add it to the undisclosed apps list
    has_bt = any(p in risk_categories["bluetooth"] for p in perms)
    mentions_bt = "bluetooth" in policy

    has_net = any(p in risk_categories["network"] for p in perms)
    mentions_net = "network" in policy

    if has_bt and not mentions_bt:
        bluetooth_undisclosed.append(data["title"])

    if has_net and not mentions_net:
      network_apps.append(data["title"])


print("Bluetooth Non-Disclosure Cases:")
print(f"{len(bluetooth_undisclosed)} / {len(flag_dict)}")

print("\nNetwork Non-Disclosure Cases:")
print(f"{len(network_undisclosed)} / {len(flag_dict)}")







Apps that ask for Network permission(s): 608 / 635 = 0.9574803149606299

Network Permission Rate: 95.75%
Bluetooth Non-Disclosure Cases:
43 / 635

Network Non-Disclosure Cases:
0 / 635


In [ ]:
#NLP and BERT integration
#Bluetooth Disclosure = policy mentions scanning, connecting, pairing,
#or communicating with nearby devices wirelessly, regardless of the keyword "Bluetooth" being mentioned

BLUETOOTH_TERMS = [
  "bluetooth", "ble", "bluetooth low energy",
  "nearby devices", "device discovery", "scan for devices",
  "scanning for devices", "pairing", "paired",
  "ibeacon", "eddystone", "beacon"
]

bt_disclosed = []

bluetooth_undisclosed = []
for app_id in bluetooth_apps:
    data = flag_dict.get(app_id, {})
    policy = (data.get("policy_text", "") or "").lower()

    if any(term in policy for term in BLUETOOTH_TERMS):
        bt_disclosed.append(app_id)
    else:
        bluetooth_undisclosed.append(app_id)

print("apps:", len(bluetooth_apps))
print("undisclosed:", len(bluetooth_undisclosed))
print("Disclosed:", len(bt_disclosed))






apps: 43
undisclosed: 43
Disclosed: 0


In [ ]:
!pip install selenium webdriver-manager lxml beautifulsoup4 requests


In [ ]:
# @title


#Chinni Version -- Issues with GECKO_PATH and PROFILE_PATH

import time
import csv
from selenium import webdriver
from selenium.webdriver.firefox.service import Service as FirefoxService
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.common.exceptions import NoSuchElementException, TimeoutException

# ==========================================
# CONFIGURATION
# ==========================================
GECKO_PATH = '/home/jinschmitt/Desktop/voice_based/Italy/geckodriver'
PROFILE_PATH = '/home/jinschmitt/Desktop/voice_based/Italy/firefox_profile'
OUTPUT_CSV = "all_categories_multiple_total.csv"

# LIST OF URLS TO SCRAPE
TARGET_URLS = [
    'https://play.google.com/store/apps/category/FAMILY',
    'https://play.google.com/store/apps/category/FAMILY?age=AGE_RANGE1',
    'https://play.google.com/store/apps/category/FAMILY?age=AGE_RANGE2',
    'https://play.google.com/store/apps/category/FAMILY?age=AGE_RANGE3'
]

# The sections to look for on EACH page
TARGET_SECTIONS = [
    "Recommended for you",
    "New & updated",
    "Love for the whole family",
    "Family apps",
    "Entertainment",
    "Simulation games",
    "Offline games",
    "Games you might like",
    "Stylized games",
    "Cartoony games",
    "Casual games",
    "Educational apps",
    "Animal games",
    "Fun watch apps for kids",
    "Games for kids",
    "Apps for kids",
    "Enriching games",
    "Action games for kids"
]

# ==========================================
# SETUP DRIVER
# ==========================================
options = webdriver.FirefoxOptions()
options.add_argument("-profile")
options.add_argument(PROFILE_PATH)

service = FirefoxService(executable_path=GECKO_PATH)
driver = webdriver.Firefox(service=service, options=options)
actions = ActionChains(driver)

collected_apps = set()

# ==========================================
# SCRAPING FUNCTION
# ==========================================
def scrape_specific_section(driver, section_title):
    """
    Locates a specific section header, waits for UI to settle, then scrapes fast.
    """
    print(f"   [SECTION] Looking for: '{section_title}'...")

    try:
        # 1. Find the header text
        section_header = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.XPATH, f"//span[text()='{section_title}']"))
        )

        # 2. Scroll to the section (SLOW: Ensure it's in view)
        driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", section_header)
        time.sleep(2)

        # 3. Locate the container
        target_section = section_header.find_element(By.XPATH, "./ancestor::div[@class='fM1mj'][1] | ./ancestor::section")

        # 4. Hover to trigger arrows (SLOW: Ensure arrow fades in)
        actions.move_to_element(target_section).perform()
        time.sleep(2) # Wait for arrow animation

        # 5. Carousel Loop (FAST)
        while True:
            # A. Scrape visible apps
            apps = target_section.find_elements(By.XPATH, ".//a[contains(@href, '/store/apps/details?id=')]")

            for app in apps:
                href = app.get_attribute("href")
                if href and href not in collected_apps:
                    collected_apps.add(href)

            # B. Find and Click Next (FAST)
            try:
                next_button = target_section.find_element(By.XPATH, ".//button[@aria-label='Scroll Next']")

                actions.move_to_element(next_button).perform()

                if next_button.is_displayed():
                    driver.execute_script("arguments[0].click();", next_button)
                    time.sleep(1.5) # FAST: Just enough for the carousel to slide
                else:
                    break
            except NoSuchElementException:
                break
            except Exception as e:
                print(f"      -> Warning: Issue clicking next: {e}")
                break

        print(f"      -> Finished '{section_title}'.")

    except TimeoutException:
        print(f"      -> Skipped: Could not find section '{section_title}' on this URL.")
    except Exception as e:
        print(f"      -> Error processing '{section_title}': {e}")


# ==========================================
# MAIN EXECUTION
# ==========================================
try:
    # LOOP THROUGH EACH URL
    for index, url in enumerate(TARGET_URLS):
        print(f"\n==========================================")
        print(f"PROCESSING URL {index + 1}/{len(TARGET_URLS)}: {url}")
        print(f"==========================================")

        driver.get(url)
        time.sleep(5) # Allow full page load for this URL

        # Iterate through all requested sections for this specific URL
        for section in TARGET_SECTIONS:
            scrape_specific_section(driver, section)
            time.sleep(1)

    # ==========================================
    # SAVE TO CSV
    # ==========================================
    print(f"\n==========================================")
    print(f"ALL DONE. Total unique apps collected across all URLs: {len(collected_apps)}")

    with open(OUTPUT_CSV, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(["App URL", "Package ID"])

        for url in collected_apps:
            try:
                pkg = url.split("id=")[1].split("&")[0]
            except:
                pkg = "Unknown"
            writer.writerow([url, pkg])

    print(f"Data saved to {OUTPUT_CSV}")

except Exception as e:
    print(f"CRITICAL ERROR: {e}")

finally:
    driver.quit()

# Task
Load the app data from "family_apps_strict_enriched.json" into a pandas DataFrame. Group the data by 'genre' and sample a total of 50-100 apps, ensuring a diverse mix of categories (e.g., Education, Games). Select the columns 'title' (rename to App Name), 'appId' (rename to Package ID), and 'privacyPolicy' (rename to Privacy Policy URL), then save this sampled dataframe to a CSV file named "sampled_apps_list.csv". Finally, confirm the file creation and display the first few rows of the sampled data.

## Load App Data

### Subtask:
Load the app data from 'family_apps_strict_enriched.json' into a pandas DataFrame.


**Reasoning**:
Load the JSON data into a pandas DataFrame as requested by the subtask instructions.



In [ ]:
import json
import pandas as pd

# Open the file and load the data
with open("family_apps_strict_enriched.json", "r") as f:
    data = json.load(f)

# Convert to DataFrame
df = pd.json_normalize(data)

# Display the first 5 rows and the shape
print("Shape of the DataFrame:", df.shape)
df.head()